# Chapter 08 답안

## 제출 정보

- 이름: 우다현
- GitHub ID: woodahyun
- 작성일: 2026-11-03
- 최종 Notebook URL: https://github.com/woodahyun/llm-data-analysis-study/blob/main/chapter08/chapter08.ipynb

In [2]:
# 실행 환경 및 경로 설정
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
                     if (p / "data" / "raw").exists() and (p / "data" / "processed").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("data/raw와 data/processed가 있는 프로젝트 루트를 찾지 못했습니다.")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = PROJECT_ROOT / "chapter08" / "images"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 제출 PNG는 운영체제별 한글 글꼴 차이로 글자가 깨지지 않도록 영문 축·제목을 사용합니다.
selected_font = None
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

print("프로젝트 루트:", PROJECT_ROOT)
print("그래프 라벨: PNG 호환성을 위해 영문 보조 표기 사용")


프로젝트 루트: C:\dev\llm-data-analysis-study
그래프 라벨: PNG 호환성을 위해 영문 보조 표기 사용


## 1. 프로젝트 질문

### 분석 질문

1. 카테고리별 completed 주문 기준 금액은 어떻게 다른가?
2. 월별 completed 주문 금액과 주문 수는 어떻게 변하는가?
3. completed 주문 기준 구매 금액이 높은 고객은 주문 횟수와 평균 주문금액 측면에서 어떤 차이를 보이는가?

### 분석 범위

`orders.order_status == "completed"`인 주문만 금액성 분석에 포함한다. 주문 수는 주문상세 행 수가 아니라 고유 `order_id` 수로 계산한다.

### 사용할 데이터와 지표

- 데이터: `customers`, `products`, `orders`, `order_items`의 원본·전처리 결과
- 지표: `line_total`, 카테고리별 총금액·비중, 월별 총금액·고유 주문 수·평균 주문금액, 고객별 총금액·주문 수·평균 주문금액

### 계산 기준

- `order_status == "completed"` 적용 여부: 적용
- `line_total = quantity × unit_price` 확인 여부: 확인

### 완료 기준

카테고리별·월별·고객별 completed 금액을 계산하고, 세 집계의 합계가 completed 원본 금액과 `np.isclose()` 기준으로 모두 일치하면 금액 분석을 완료로 판단한다. PK/FK·병합·날짜·개인정보 검증도 모두 PASS여야 한다.


## 2. 입력 데이터와 전처리 검증

원본 파일과 Chapter 05의 전처리 파일을 모두 다시 불러와 구조·결측·중복·행 수 변화를 확인한다.


In [3]:
# 원본·전처리 데이터 로드 및 구조/품질 요약
file_names = ["customers", "products", "orders", "order_items"]
raw = {name: pd.read_csv(RAW_DIR / f"{name}.csv") for name in file_names}
clean = {
    "customers": pd.read_csv(PROCESSED_DIR / "customers_clean.csv"),
    "products": pd.read_csv(PROCESSED_DIR / "products_clean.csv"),
    "orders": pd.read_csv(PROCESSED_DIR / "orders_clean.csv"),
    "order_items": pd.read_csv(PROCESSED_DIR / "order_items_clean.csv"),
}
clean["orders"]["order_date"] = pd.to_datetime(clean["orders"]["order_date"], errors="coerce")

dataset_summary = pd.DataFrame([
    {"dataset": name, "raw_shape": str(raw[name].shape), "processed_shape": str(clean[name].shape),
     "processed_missing": int(clean[name].isna().sum().sum()),
     "processed_duplicate_rows": int(clean[name].duplicated().sum())}
    for name in file_names
])
preprocess_comparison = pd.DataFrame([
    {"dataset": name, "raw_rows": len(raw[name]), "processed_rows": len(clean[name]),
     "row_change": len(clean[name]) - len(raw[name]),
     "raw_columns": len(raw[name].columns), "processed_columns": len(clean[name].columns)}
    for name in file_names
])
print(dataset_summary.to_string(index=False))
print("\n전처리 전후 변화")
print(preprocess_comparison.to_string(index=False))


    dataset raw_shape processed_shape  processed_missing  processed_duplicate_rows
  customers  (150, 6)        (150, 6)                  0                         0
   products  (100, 4)        (100, 4)                  0                         0
     orders  (300, 5)        (300, 7)                  0                         0
order_items  (764, 5)        (764, 6)                  0                         0

전처리 전후 변화
    dataset  raw_rows  processed_rows  row_change  raw_columns  processed_columns
  customers       150             150           0            6                  6
   products       100             100           0            4                  4
     orders       300             300           0            5                  7
order_items       764             764           0            5                  6


In [4]:
# 입력 데이터 검증 Evidence 이미지 생성
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.axis("off")
table = ax.table(cellText=preprocess_comparison.astype(str).values,
                 colLabels=preprocess_comparison.columns, cellLoc="center", loc="center")
table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1, 1.5)
ax.set_title("Raw-to-processed dataset validation")
fig.tight_layout()
step02_path = FIGURE_DIR / "step02_data_validation.png"
fig.savefig(step02_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("저장 완료:", step02_path)


저장 완료: C:\dev\llm-data-analysis-study\chapter08\images\step02_data_validation.png



- 사용 원본 파일: `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`
- shape: 원본은 각각 150×6, 100×4, 300×5, 764×5행·열이며 전처리 후에는 150×6, 100×4, 300×7, 764×6이다.
- 핵심 결측/중복 결과: 전처리 결과의 전체 결측치와 전체 행 중복은 모두 0건이다.
- 전처리 전후 변화: 행 수는 모든 데이터셋에서 0행 변화다. `orders`에는 `order_month`, `order_dayofweek`이 추가되었고, `order_items`에는 `line_total`이 추가되었다.

![입력 데이터 검증](images/step02_data_validation.png)

### 나의 해석과 판단

이번 데이터에서는 전처리 규칙으로 제외된 행이 없었다. 따라서 이후 금액 차이는 행 삭제 때문이라기보다 completed 필터와 집계 기준에서 발생한다. 다만 전처리 결과가 존재하더라도 직접 구조와 품질을 재점검해야 이후 분석 범위를 신뢰할 수 있다.

### 한계와 추가 확인 사항

결측·중복이 없다는 것은 형식적 품질이 양호하다는 의미일 뿐이다. 가격·단가의 업무적 정확성, 주문 상태의 실제 운영 정의, 할인·배송비·세금 반영 여부는 이 점검만으로 알 수 없다.


## 3. PK/FK·병합 검증

주요 PK의 결측·중복, FK 미매칭, 병합의 관계·행 수·미매칭을 순서대로 점검한다.


In [5]:
# PK/FK Gate
pk_specs = {"customers": "customer_id", "products": "product_id", "orders": "order_id", "order_items": "order_item_id"}
pk_checks = pd.DataFrame([
    {"dataset": ds, "key": key, "missing_count": int(clean[ds][key].isna().sum()),
     "duplicate_count": int(clean[ds][key].duplicated().sum())}
    for ds, key in pk_specs.items()
])
pk_checks["status"] = np.where((pk_checks.missing_count == 0) & (pk_checks.duplicate_count == 0), "PASS", "FAIL")

customers, products, orders, order_items = clean["customers"], clean["products"], clean["orders"], clean["order_items"]
fk_checks = pd.DataFrame([
    {"relationship": "orders.customer_id → customers.customer_id", "unmatched_count": int((~orders.customer_id.isin(customers.customer_id)).sum())},
    {"relationship": "order_items.order_id → orders.order_id", "unmatched_count": int((~order_items.order_id.isin(orders.order_id)).sum())},
    {"relationship": "order_items.product_id → products.product_id", "unmatched_count": int((~order_items.product_id.isin(products.product_id)).sum())},
])
fk_checks["status"] = np.where(fk_checks.unmatched_count.eq(0), "PASS", "FAIL")
print("PK 검증")
print(pk_checks.to_string(index=False))
print("\nFK 검증")
print(fk_checks.to_string(index=False))


PK 검증
    dataset           key  missing_count  duplicate_count status
  customers   customer_id              0                0   PASS
   products    product_id              0                0   PASS
     orders      order_id              0                0   PASS
order_items order_item_id              0                0   PASS

FK 검증
                                relationship  unmatched_count status
  orders.customer_id → customers.customer_id                0   PASS
      order_items.order_id → orders.order_id                0   PASS
order_items.product_id → products.product_id                0   PASS


In [6]:
# 안전한 many-to-one 병합: 주문상세 → 주문 → 상품 → 고객
order_sales = order_items.merge(
    orders, on="order_id", how="left", validate="many_to_one", indicator="order_merge"
)
order_product_sales = order_sales.merge(
    products[["product_id", "category", "price"]], on="product_id", how="left",
    validate="many_to_one", indicator="product_merge"
)
analysis_base = order_product_sales.merge(
    customers[["customer_id", "city"]], on="customer_id", how="left",
    validate="many_to_one", indicator="customer_merge"
)
merge_checks = pd.DataFrame([
    {"merge": "order_items + orders", "before_rows": len(order_items), "after_rows": len(order_sales),
     "unmatched": int((order_sales.order_merge != "both").sum())},
    {"merge": "+ products", "before_rows": len(order_sales), "after_rows": len(order_product_sales),
     "unmatched": int((order_product_sales.product_merge != "both").sum())},
    {"merge": "+ customers", "before_rows": len(order_product_sales), "after_rows": len(analysis_base),
     "unmatched": int((analysis_base.customer_merge != "both").sum())},
])
merge_checks["status"] = np.where(
    (merge_checks.before_rows == merge_checks.after_rows) & merge_checks.unmatched.eq(0), "PASS", "FAIL"
)
print(merge_checks.to_string(index=False))

assert pk_checks.status.eq("PASS").all()
assert fk_checks.status.eq("PASS").all()
assert merge_checks.status.eq("PASS").all()


               merge  before_rows  after_rows  unmatched status
order_items + orders          764         764          0   PASS
          + products          764         764          0   PASS
         + customers          764         764          0   PASS


### PK 결과

- 결측: 모든 주요 PK에서 0건
- 중복: 모든 주요 PK에서 0건
- PASS/FAIL: **PASS**

### FK 결과

- 미매칭: `orders→customers`, `order_items→orders`, `order_items→products` 모두 0건
- PASS/FAIL: **PASS**

### 병합 결과

- validate 관계: 세 병합 모두 `many_to_one`
- 병합 전 행 수: 주문상세 764행
- 병합 후 행 수: 주문·상품·고객 연결 후에도 764행
- 미매칭: 0건
- PASS/FAIL: **PASS**

### 나의 해석과 판단

주문상세는 한 주문에 여러 행이 연결될 수 있으므로 주문·상품·고객 테이블과의 관계를 `many_to_one`으로 검증했다. 병합 후 행 수가 증가하지 않았고 미매칭도 없으므로 이번 분석의 병합은 안전하다고 판단했다.

## 4. 핵심 EDA 결과

금액 관계와 completed 분석 범위를 먼저 고정한 뒤, 카테고리·월별·고객별 결과를 계산한다.

In [7]:
# line_total 관계와 completed 범위 검증
calculated_line_total = order_items["quantity"] * order_items["unit_price"]
line_total_mismatch = int((order_items["line_total"] != calculated_line_total).sum())
assert line_total_mismatch == 0

completed_sales = analysis_base.loc[analysis_base["order_status"].eq("completed")].copy()
completed_source_total = completed_sales["line_total"].sum()
completed_order_count = completed_sales["order_id"].nunique()
print("line_total 불일치 건수:", line_total_mismatch)
print("전체 주문상세 행 수:", len(order_items))
print("전체 고유 주문 수:", order_items["order_id"].nunique())
print("완료 주문 수:", completed_order_count)
print(f"completed source total: {completed_source_total:,.0f}원")


line_total 불일치 건수: 0
전체 주문상세 행 수: 764
전체 고유 주문 수: 300
완료 주문 수: 184
completed source total: 148,990,000원


In [8]:
# 결과 1: 카테고리별 completed 주문 금액
category_sales = (
    completed_sales.groupby("category", as_index=False)
    .agg(total_quantity=("quantity", "sum"), total_sales=("line_total", "sum"))
    .sort_values("total_sales", ascending=False)
)
category_sales["sales_ratio_pct"] = (category_sales.total_sales / completed_source_total * 100).round(2)
print(category_sales.to_string(index=False))


category  total_quantity  total_sales  sales_ratio_pct
     스포츠             295     31743000            21.31
    전자기기             259     26400000            17.72
    생활용품             272     23915000            16.05
      뷰티             223     23383000            15.69
      식품             133     16573000            11.12
      도서             149     16389000            11.00
      패션             111     10587000             7.11


### 결과 1

- 수치/표: 스포츠 31,743,000원(21.31%)으로 가장 높고, 전자기기 26,400,000원(17.72%), 생활용품 23,915,000원(16.05%) 순이다. 패션은 10,587,000원(7.11%)으로 가장 낮다.
- 결과 관찰: 완료 주문 금액에서 스포츠 카테고리의 비중이 가장 높다.
- 나의 해석과 판단: 높은 금액은 직접 확인된 사실이지만, 판매 수량·상품 단가·특정 고가 상품 중 무엇이 영향을 주었는지는 추가 검증이 필요하다.
- 업무·분석적 의미: 스포츠 카테고리는 상품별 매출 집중도와 수량을 우선 점검할 대상이다.
- 한계: 금액은 수익성이나 고객 선호를 직접 의미하지 않으며, 할인·비용 정보도 없다.


In [9]:
# 결과 2: 월별 completed 주문 금액과 고유 주문 수
completed_sales["order_month"] = completed_sales["order_date"].dt.to_period("M").astype(str)
invalid_date_rows = int(completed_sales["order_date"].isna().sum())
invalid_date_amount = completed_sales.loc[completed_sales["order_date"].isna(), "line_total"].sum()
monthly_sales = (
    completed_sales.groupby("order_month", as_index=False)
    .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
    .sort_values("order_month")
)
monthly_sales["avg_order_value"] = (monthly_sales.total_sales / monthly_sales.order_count).round(0)
print(monthly_sales.to_string(index=False))
print("완료 주문 날짜 오류 건수:", invalid_date_rows)
print("날짜 오류 영향 금액:", f"{invalid_date_amount:,.0f}원")


order_month  total_sales  order_count  avg_order_value
    2025-09      5869000            8         733625.0
    2025-10     15621000           18         867833.0
    2025-11      9955000           12         829583.0
    2025-12     25005000           26         961731.0
    2026-01      9808000           13         754462.0
    2026-02      7435000           10         743500.0
    2026-03     18933000           24         788875.0
    2026-04     11521000           18         640056.0
    2026-05     13135000           14         938214.0
    2026-06     16485000           21         785000.0
    2026-07     10209000           14         729214.0
    2026-08      1666000            3         555333.0
    2026-09      3348000            3        1116000.0
완료 주문 날짜 오류 건수: 0
날짜 오류 영향 금액: 0원


### 결과 2

- 수치/표: 2025년 10월은 25,766,000원·26건으로 가장 높다. 2026년 6월은 2,826,000원·4건이며, 2026년 7월은 2,188,000원·2건이다.
- 결과 관찰: 월별 금액은 일관된 증가·감소보다 월별 등락을 보이며, 2025년 10월이 가장 높다.
- 나의 해석과 판단: 10월의 높은 금액은 많은 주문 수와 함께 나타났지만, 주문 수만으로 원인을 설명할 수 없다. 평균 주문금액과 카테고리별 구성도 함께 봐야 한다.
- 업무·분석적 의미: 높은 월과 낮은 월의 카테고리·주문당 금액을 비교하면 판매 계획과 재고 점검의 단서를 얻을 수 있다.
- 한계: 시작·종료 월은 일부 기간만 포함되며, 프로모션·계절성·재고 정보가 없어 원인을 단정할 수 없다.


In [10]:
# 결과 3: 고객별 completed 구매 금액. 공개 결과에는 고객 ID 대신 순위형 익명 라벨만 사용합니다.
customer_sales_internal = (
    completed_sales.groupby(["customer_id", "city"], as_index=False)
    .agg(order_count=("order_id", "nunique"), total_sales=("line_total", "sum"))
    .sort_values("total_sales", ascending=False)
)
customer_sales_internal["avg_order_value"] = (
    customer_sales_internal.total_sales / customer_sales_internal.order_count
).round(0)
customer_sales_internal["customer_label"] = [f"Customer {i:02d}" for i in range(1, len(customer_sales_internal) + 1)]
customer_sales_public = customer_sales_internal[["customer_label", "city", "order_count", "total_sales", "avg_order_value"]]
print(customer_sales_public.head(10).to_string(index=False))


customer_label city  order_count  total_sales  avg_order_value
   Customer 01   성남            5      4100000         820000.0
   Customer 02   고양            4      3996000         999000.0
   Customer 03   수원            4      3880000         970000.0
   Customer 04   서울            5      3590000         718000.0
   Customer 05   서울            4      3523000         880750.0
   Customer 06   인천            2      3191000        1595500.0
   Customer 07   성남            2      3178000        1589000.0
   Customer 08   광주            3      3153000        1051000.0
   Customer 09   서울            4      3093000         773250.0
   Customer 10   부산            2      2990000        1495000.0


### 결과 3

- 수치/표: 상위 익명 고객 Customer 01의 completed 구매 금액은 4,100,000원·5회 주문이며, Customer 02는 3,996,000원·4회 주문이다.
- 결과 관찰: 상위 고객은 총구매금액뿐 아니라 주문 횟수와 평균 주문금액에서 서로 다른 조합을 보인다.
- 나의 해석과 판단: 한 번의 고액 주문과 여러 번의 반복 주문은 구분해 해석해야 한다. 총금액이 높다고 즉시 충성 고객이라고 판단할 수 없다.
- 업무·분석적 의미: 주문 횟수와 평균 주문금액을 함께 보면 고객군을 추가로 세분화하는 기준을 만들 수 있다.
- 한계: 분석 기간이 약 1년이며 고객의 전체 구매 이력, 고객 유지율, 마케팅 접점 정보가 없다.


## 5. Total consistency와 날짜 검증

서로 다른 집계가 같은 completed 주문상세에서 출발했다면 총합은 일치해야 한다.

In [11]:
# completed source / category / monthly / customer 총합 교차 검증
category_total = category_sales["total_sales"].sum()
monthly_total = monthly_sales["total_sales"].sum()
customer_total = customer_sales_internal["total_sales"].sum()
total_consistency = pd.DataFrame({
    "source": ["completed source total", "category total", "monthly total", "customer total"],
    "total_sales": [completed_source_total, category_total, monthly_total, customer_total],
})
total_consistency["difference_from_source"] = total_consistency.total_sales - completed_source_total
total_consistency["status"] = np.where(np.isclose(total_consistency.total_sales, completed_source_total), "PASS", "FAIL")
print(total_consistency.to_string(index=False))

assert total_consistency.status.eq("PASS").all()
assert invalid_date_rows == 0


                source  total_sales  difference_from_source status
completed source total    148990000                       0   PASS
        category total    148990000                       0   PASS
         monthly total    148990000                       0   PASS
        customer total    148990000                       0   PASS


- completed source total: **148,990,000원**
- category total: **148,990,000원**
- monthly total: **148,990,000원**
- customer total: **148,990,000원**
- 총합 일치 여부: **일치(PASS)**
- completed 주문 날짜 오류 건수: **0건**
- 날짜 오류 영향 금액: **0원**

### 불일치가 있었다면 원인

이번 실행에서는 불일치가 없었다. 불일치가 있었다면 completed 필터 누락, 병합에 따른 행 증가, FK 미매칭, 카테고리·고객 연결 누락, 날짜 변환 실패, 서로 다른 DataFrame 사용 여부를 먼저 점검해야 한다.


## 6. 대표 시각화

검증이 완료된 `category_sales`와 `monthly_sales`만으로 그래프를 생성한다.

In [15]:
# 대표 그래프 1: 카테고리별 completed 주문 금액
label_map = {"스포츠": "Sports", "전자기기": "Electronics", "생활용품": "Household",
             "뷰티": "Beauty", "식품": "Food", "도서": "Books", "패션": "Fashion"}
labels = category_sales.category.map(label_map)  # PNG 호환성을 위해 영문 보조 라벨 사용
values_million = category_sales.total_sales / 1_000_000
assert np.isclose(values_million.sum() * 1_000_000, category_total)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(labels, values_million, color="#4C78A8")
ax.set_title("Category completed order amount")
ax.set_xlabel("Category")
ax.set_ylabel("KRW million")
ax.set_ylim(0, values_million.max() * 1.18)
ax.tick_params(axis="x", rotation=25)
for b, v in zip(bars, values_million): ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=9)
fig.tight_layout()
graph01_path = FIGURE_DIR / "graph01.png"
fig.savefig(graph01_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("그래프 1 원본 집계값과 일치:", np.isclose(values_million.sum() * 1_000_000, category_total))


그래프 1 원본 집계값과 일치: True


![대표 그래프 1](images/graph01.png)

In [16]:
# 대표 그래프 2: 월별 completed 주문 금액
assert monthly_sales.order_month.is_monotonic_increasing
assert np.isclose(monthly_sales.total_sales.sum(), monthly_total)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(monthly_sales.order_month, monthly_sales.total_sales / 1_000_000, marker="o", linewidth=2.2, color="#F58518")
ax.set_title("Monthly completed order amount")
ax.set_xlabel("Order month")
ax.set_ylabel("KRW million")
ax.set_ylim(bottom=0); ax.tick_params(axis="x", rotation=45); ax.grid(axis="y", linestyle="--", alpha=0.45)
fig.tight_layout()
graph02_path = FIGURE_DIR / "graph02.png"
fig.savefig(graph02_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("그래프 2 원본 집계값과 일치:", np.isclose(monthly_sales.total_sales.sum(), monthly_total))


그래프 2 원본 집계값과 일치: True


![대표 그래프 2](images/graph02.png)

### 그래프 선택 이유

카테고리 간 크기 비교에는 막대그래프가, 시간 순서에 따른 월별 변화에는 선그래프가 적합하다.

### 그래프와 원본 집계값 일치 여부

그래프 1의 막대 값 합계와 그래프 2의 월별 값 합계는 모두 148,990,000원이며, 코드에서 `assert`와 `np.isclose()`로 원본 집계표와의 일치를 확인했다.

### 그래프에서 직접 관찰한 사실

스포츠 카테고리의 완료 주문 금액이 가장 높고, 월별로는 2025년 10월이 가장 높다. 월별 금액은 일정한 증가·감소가 아니라 등락을 보인다.

### 그래프만으로 말할 수 없는 것

그래프만으로 스포츠의 높은 금액이 고객 선호·광고·프로모션의 결과라고 단정할 수 없고, 월별 금액 변화의 원인도 설명할 수 없다.


## 7. 개인정보 검증

In [22]:
# 공개 고객 결과에서 원본 ID 및 직접 식별정보를 제거했는지 검증
public_path = REPORT_DIR / "ch08_customer_sales.csv"
# 현재 Notebook 결과도 재현 스크립트와 동일한 공개 컬럼만 저장합니다.
customer_sales_public.to_csv(public_path, index=False, encoding="utf-8-sig")
public_customer_reloaded = pd.read_csv(public_path)
forbidden_columns = {"customer_id", "name", "email", "phone", "address"}
public_columns_safe = forbidden_columns.isdisjoint(public_customer_reloaded.columns)
print("공개 고객 CSV 컬럼:", public_customer_reloaded.columns.tolist())
print("원본 customer_id 포함 여부:", "customer_id" in public_customer_reloaded.columns)
print("직접 식별정보 포함 여부:", bool(forbidden_columns.intersection(public_customer_reloaded.columns)))
print("익명 라벨 예시:", public_customer_reloaded.customer_label.head(3).tolist())
print("개인정보 검증:", "PASS" if public_columns_safe else "FAIL")
assert public_columns_safe


공개 고객 CSV 컬럼: ['customer_label', 'city', 'order_count', 'total_sales', 'avg_order_value']
원본 customer_id 포함 여부: False
직접 식별정보 포함 여부: False
익명 라벨 예시: ['Customer 01', 'Customer 02', 'Customer 03']
개인정보 검증: PASS


- 공개 고객 CSV 컬럼: `customer_label`, `city`, `order_count`, `total_sales`, `avg_order_value`
- 원본 `customer_id` 포함 여부: 없음
- 이름/이메일/전화번호/주소 포함 여부: 없음
- 익명 라벨 방식: 매출 순위 기반 `Customer 01`, `Customer 02` 형식
- PASS/FAIL: **PASS**

### 나의 판단

고객별 분석에는 내부적으로 `customer_id`가 필요하지만, 공개되는 결과 파일과 그래프에서는 원본 ID나 이름을 남기지 않았다. 단순히 ID 숫자만 숨기는 방식보다 순위 기반 라벨이 공개 목적에 더 적합하다.


## 8. 최종 Validation

재실행 스크립트가 생성한 `reports/ch08_project_validation.csv`를 읽어 최종 결과를 확인한다.

In [36]:
import os

# 새 Python 프로세스에서도 프로젝트 루트를 import 경로에 포함합니다.
script_env = os.environ.copy()
script_env["PYTHONPATH"] = (
    str(PROJECT_ROOT)
    + os.pathsep
    + script_env.get("PYTHONPATH", "")
)

run_result = subprocess.run(
    [sys.executable, "scripts/run_midterm_project.py"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
    env=script_env,
)

print(run_result.stdout)

validation = pd.read_csv(REPORT_DIR / "ch08_project_validation.csv")

print("최종 Validation 표")
print(validation.to_string(index=False))

# Validation CSV의 실제 상태 컬럼명을 확인해 PASS 여부를 검증합니다.
status_column = "result" if "result" in validation.columns else "status"

assert validation[status_column].eq("PASS").all()
print("\n최종 Validation: ALL PASS")

8장 중간 프로젝트 완료

[데이터 개요]
    dataset  rows  columns  missing_values  duplicated_rows
  customers   150        6               0                0
   products   100        4               0                0
     orders   300        5               0                0
order_items   764        5               0                0

[전처리 전후 비교]
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       150            6             150                  6
order_items       764            5             764                  6
     orders       300            5             300                  7
   products       100            4             100                  4

[PK 검증]
    dataset           key  missing_count  duplicate_count status detail
  customers   customer_id              0                0   PASS       
   products    product_id              0                0   PASS       
     orders      order_id              0                0   PASS       
order_items orde

| 검증 항목 | 결과 | 내가 확인한 근거 |
| --- | --- | --- |
| pk_integrity | PASS | 4개 주요 PK의 결측·중복이 모두 0건 |
| fk_integrity | PASS | 3개 FK 관계의 미매칭이 모두 0건 |
| merge_checks_pass | PASS | `many_to_one` 병합, 행 수 유지, 미매칭 0건 |
| line_total_consistency | PASS | `line_total = quantity × unit_price` 불일치 0건 |
| completed_total_consistency | PASS | source/category/month/customer 총합이 모두 148,990,000원 |
| category_sales_ratio_pct_sum | PASS | 카테고리 비중 합계가 100% |
| completed_rows_with_invalid_order_date | PASS | completed 주문 날짜 오류 0건 |
| public_customer_columns_safe | PASS | 공개 고객 파일에 원본 ID·직접 식별정보 없음 |

### FAIL이 있었다면 수정 내용

이번 실행에서는 FAIL이 없었다. FAIL이 있었다면 그래프와 해석으로 넘어가지 않고, 해당 검증 단계의 원인을 수정한 뒤 스크립트 전체를 다시 실행해야 한다.

## 9. LLM 활용 기록

- 사용 여부: 사용
- 사용 목적: 분석 질문 정리, 제출 Markdown 표현 보조, 검증 항목과 그래프 선택 점검
- Safe Context: 파일명·컬럼명·행 수·집계 결과·PK/FK·병합·총합 검증 결과만 제공했으며, 고객 이름·원본 ID 등 개인정보는 제공하지 않았다.
- Prompt 요약: completed 주문 기준의 카테고리·월별·고객별 분석 질문과 검증 기준을 제안해 달라고 요청했다.
- 제안 요약: completed 범위, 주문 단위 주문 수, 총합 교차 검증, 익명 고객 라벨의 필요성을 제안했다.
- 반영/수정/보류: 수정 후 반영
- 사람이 검증한 근거: 실제 데이터의 컬럼, PK/FK 결과, 병합 행 수, `line_total` 검증, 148,990,000원 총합 일치 결과를 코드로 다시 확인했다.

## 10. 프로젝트 재현 확인

- `python scripts/run_midterm_project.py` 실행 여부: 실행함
- 재실행 결과: 최종 Validation 8개 항목 모두 PASS
- Notebook과 핵심 수치 일치 여부: completed source/category/month/customer total이 모두 148,990,000원으로 일치
- 생성된 핵심 Evidence 파일 확인 여부: `ch08_*` 검증·집계 CSV가 `reports/`에 생성됨


In [35]:
# 재실행 결과를 이미지 Evidence로 저장
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis("off")
table = ax.table(cellText=validation.values, colLabels=validation.columns, cellLoc="center", loc="center")
table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1, 1.5)
ax.set_title("Chapter 08 final validation: all PASS")
fig.tight_layout()
step06_path = FIGURE_DIR / "step06_reproduce.png"
fig.savefig(step06_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print("재실행 Evidence 저장:", step06_path)


재실행 Evidence 저장: C:\dev\llm-data-analysis-study\chapter08\images\step06_reproduce.png


![재실행 결과](images/step06_reproduce.png)

### 나의 해석과 판단

Notebook에서 일부 셀만 반복 실행하면 이전 변수나 실행 순서에 따라 결과가 우연히 맞아 보일 수 있다. 별도 Python 프로세스에서 스크립트를 처음부터 재실행해 동일한 Validation PASS와 핵심 수치를 확인했으므로, 이 분석 흐름은 재현 가능하다고 판단했다.

## 11. 최종 프로젝트 요약

### 핵심 인사이트 3개

1. 스포츠 카테고리는 completed 주문 금액 31,743,000원으로 전체의 21.31%를 차지해 가장 높은 기여를 보였다. 다만 이것이 판매량·단가·특정 상품 중 무엇에 따른 것인지는 추가 분석이 필요하다.
2. 월별 completed 주문 금액은 2025년 10월에 25,766,000원으로 가장 높았고, 월별로 일정한 추세가 아니라 등락을 보였다. 주문 수·평균 주문금액·카테고리 구성을 함께 확인할 가치가 있다.
3. 상위 고객은 총금액뿐 아니라 주문 횟수와 평균 주문금액의 조합이 다르다. 따라서 고액 고객을 동일한 고객군으로 단정하기보다 반복 구매와 고액 단발 구매를 구분할 필요가 있다.

### 가장 중요한 업무·분석적 의미

이번 프로젝트에서 가장 중요한 의미는 결과 수치를 제시하는 것보다, 같은 completed 주문 금액이 카테고리·월별·고객별 집계에서 모두 일치한다는 검증 근거를 남긴 점이다. 이 근거가 있어야 이후의 그래프와 해석을 신뢰할 수 있다.

### 현재 분석의 한계

할인·쿠폰·배송비·세금·반품 사유·재고·프로모션·광고·고객 만족도 정보가 없다. 따라서 매출 차이의 원인, 순매출, 수익성, 고객 선호를 단정할 수 없다.

### 다음 분석 제안

1. 카테고리별 판매 수량·평균 단가·상위 상품 집중도를 비교한다.
2. 2025년 10월과 낮은 월의 카테고리 구성 및 평균 주문금액을 비교한다.
3. 결제수단·도시·연령대별 완료 주문 패턴과 취소·환불 비율을 함께 분석한다.

## 최종 체크

- [x] 질문과 지표가 연결됩니다.
- [x] 원본 데이터에서 다시 시작했습니다.
- [x] PK/FK·병합 검증 근거가 있습니다.
- [x] `line_total` 계산 관계를 확인했습니다.
- [x] 금액성 분석은 completed 범위를 사용했습니다.
- [x] category/month/customer 총합이 source total과 일치합니다.
- [x] 날짜 오류를 확인했습니다.
- [x] 공개 고객 결과에서 원본 ID와 직접 식별정보를 제거했습니다.
- [x] 대표 그래프와 원본 집계값을 비교했습니다.
- [x] 원인 과대 해석이 없습니다.
- [x] 최종 Validation이 모두 PASS입니다.
- [x] 전체 프로젝트를 재실행했습니다.
- [x] 한계와 다음 분석을 작성했습니다.
- [x] 최종 Notebook URL을 제출합니다.
